# PetFinder · v5b · Modelado con embeddings

**Qué agrega sobre v4.** Los 384 números por mascota que calculó v5a, reducidos a 20 componentes con SVD y pegados a `train` y `test` como 20 columnas más, igual que las de TF-IDF.

**Pregunta que responde.** ¿Un embedding le gana a TF-IDF? ¿O suman juntos? Se corren tres variantes con el mismo regresor: solo TF-IDF (v4), solo embeddings, y los dos.

**Cómo correrla en Kaggle.** Copy & Edit. *Add Input → Your Work → Notebooks → v5a* (la que generó los parquet). **Internet off.** Verificar en la celda de carga que la ruta `EMB_INPUT` coincide con el nombre de tu notebook v5a. *Save & Run All → Output → Submit*.

In [ ]:
import os
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier, LGBMRegressor
from sklearn.model_selection import KFold, StratifiedGroupKFold
from sklearn.metrics import cohen_kappa_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from tqdm.notebook import tqdm

INPUT = os.environ.get("PETFINDER_INPUT", "/kaggle/input/petfinder-adoption-prediction")
if not os.path.exists(INPUT):   # las notebooks creadas por la API montan la competencia bajo /kaggle/input/competitions/
    INPUT = "/kaggle/input/competitions/petfinder-adoption-prediction"
SEED = 42

In [ ]:
train = pd.read_csv(f"{INPUT}/train/train.csv", index_col="PetID")
test = pd.read_csv(f"{INPUT}/test/test.csv", index_col="PetID")
ss = pd.read_csv(f"{INPUT}/test/sample_submission.csv", index_col="PetID")

label = "AdoptionSpeed"

In [ ]:
def agregar_features(data):
    """Las tres features de la consigna. Se aplica igual a train y a test."""
    data = data.copy()
    # 1) cuántas publicaciones tiene el rescatista (calculado dentro de cada dataset)
    data["rescuer_count"] = data["RescuerID"].map(data["RescuerID"].value_counts())
    # 2) la mascota no tiene nombre (vacío, "No Name", "No Name Yet"...)
    nombre = data["Name"].fillna("").str.lower().str.replace(" ", "", regex=False)
    data["sin_nombre"] = ((nombre == "") | nombre.str.startswith("noname")).astype(int)
    # 3) largo de la descripción
    desc = data["Description"].fillna("").astype(str)
    data["desc_len"] = desc.str.len()
    data["desc_words"] = desc.str.split().str.len()
    return data


train = agregar_features(train)
test = agregar_features(test)

X_train = train.drop(label, axis=1).select_dtypes(exclude="O")
y_train = train[label]
X_test = test[X_train.columns]
groups = train["RescuerID"]   # para que cada rescatista caiga en un solo fold

print(X_train.shape, X_test.shape)

## Texto con TF-IDF, igual que v4

In [ ]:
N_SVD = 20   # componentes que entran al modelo como features

desc_train = train["Description"].fillna("").astype(str)
desc_test = test["Description"].fillna("").astype(str)
desc_all = pd.concat([desc_train, desc_test])

# TF-IDF se ajusta sobre todas las descripciones, train y test. Es no supervisado (no mira el target),
# por eso se acepta; pero conviene decirlo: el vocabulario "vio" el test.
tfidf = TfidfVectorizer(min_df=3, max_features=20000, ngram_range=(1, 2), sublinear_tf=True, strip_accents="unicode")
matriz = tfidf.fit_transform(desc_all)
print("matriz TF-IDF:", matriz.shape)

svd = TruncatedSVD(n_components=N_SVD, random_state=SEED)
Z = svd.fit_transform(matriz)
print("varianza explicada por las", N_SVD, "componentes:", round(svd.explained_variance_ratio_.sum(), 3))

cols_svd = [f"tfidf_svd_{i}" for i in range(N_SVD)]
Z_train = pd.DataFrame(Z[: len(train)], index=train.index, columns=cols_svd)
Z_test = pd.DataFrame(Z[len(train):], index=test.index, columns=cols_svd)


def stats_texto(data):
    """Estadísticas simples de la descripción: baratas y accionables para el refugio."""
    data = data.copy()
    desc = data["Description"].fillna("").astype(str)
    data["desc_upper_ratio"] = desc.str.count(r"[A-Z]") / (desc.str.len() + 1)
    data["desc_exclam"] = desc.str.count("!")
    data["desc_vacia"] = (desc.str.strip() == "").astype(int)
    return data


train = stats_texto(train).join(Z_train)
test = stats_texto(test).join(Z_test)

# las mismas líneas de siempre: ahora X_train incluye las estadísticas y las 20 componentes
X_train = train.drop(label, axis=1).select_dtypes(exclude="O")
y_train = train[label]
X_test = test[X_train.columns]
cols_sin_texto = [c for c in X_train.columns if not c.startswith("tfidf_svd_")]
print(X_train.shape, X_test.shape)

## Embeddings de v5a

Cambiar `EMB_INPUT` por la ruta de tu notebook v5a. Se ve en el panel *Input* de la derecha.

In [ ]:
EMB_INPUT = os.environ.get("PETFINDER_EMB", "/kaggle/input/petfinder-v5a-embeddings-preprocesado")
if not os.path.exists(EMB_INPUT):   # misma historia: las notebooks creadas por API montan los inputs bajo /kaggle/input/notebooks/
    import glob
    cand = glob.glob("/kaggle/input/**/train_emb.parquet", recursive=True)
    EMB_INPUT = os.path.dirname(cand[0]) if cand else EMB_INPUT

emb_train = pd.read_parquet(f"{EMB_INPUT}/train_emb.parquet").loc[train.index]
emb_test = pd.read_parquet(f"{EMB_INPUT}/test_emb.parquet").loc[test.index]
print("embeddings:", emb_train.shape, emb_test.shape)

N_EMB = 20
svd_emb = TruncatedSVD(n_components=N_EMB, random_state=SEED)
E = svd_emb.fit_transform(pd.concat([emb_train, emb_test]).values)
cols_emb = [f"emb_svd_{i}" for i in range(N_EMB)]
E_train = pd.DataFrame(E[: len(train)], index=train.index, columns=cols_emb)
E_test = pd.DataFrame(E[len(train):], index=test.index, columns=cols_emb)
print("varianza explicada por las", N_EMB, "componentes del embedding:", round(svd_emb.explained_variance_ratio_.sum(), 3))

# se pegan como columnas nuevas y X_train / X_test se rearman con las mismas líneas de siempre
train = train.join(E_train)
test = test.join(E_test)
X_train = train.drop(label, axis=1).select_dtypes(exclude="O")
y_train = train[label]
X_test = test[X_train.columns]

cols_tfidf = [c for c in X_train.columns if not c.startswith("emb_svd_")]      # v4: sin embeddings
cols_emb_only = [c for c in X_train.columns if not c.startswith("tfidf_svd_")] # embeddings en lugar de TF-IDF
print(X_train.shape, X_test.shape)

In [ ]:
def correr_cv(crear_modelo, X_train, y_train, X_test, groups):
    """El mismo loop del baseline, metido en una función para poder llamarlo varias veces.

    Devuelve valid_preds (una predicción por mascota de train, hecha por el modelo que NO la vio)
    y test_preds (un DataFrame con las 5 predicciones de test, una por fold).
    """
    valid_preds = []
    test_preds = []
    folds = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
    for train_index, valid_index in tqdm(folds.split(X_train, y_train, groups), total=5):
        Xt = X_train.iloc[train_index]
        yt = y_train.iloc[train_index]
        Xv = X_train.iloc[valid_index]

        model = crear_modelo()
        model.fit(Xt, yt)

        valid_preds.append(pd.Series(model.predict(Xv), index=Xv.index, name=label))
        test_preds.append(pd.Series(model.predict(X_test), index=X_test.index, name=label))

    valid_preds = pd.concat(valid_preds).loc[X_train.index]
    test_preds = pd.concat(test_preds, axis=1)
    return valid_preds, test_preds


def kappa(y_true, y_pred):
    return cohen_kappa_score(y_true, y_pred, weights="quadratic")

In [ ]:
def umbrales_por_cuantiles(pred_cont, y):
    """Cuatro cortes para que la distribución de clases predicha copie la del train."""
    acumulado = np.cumsum(np.bincount(y, minlength=5) / len(y))[:-1]   # 2,7% · 23,3% · 50,2% · 72,0%
    return np.quantile(pred_cont, acumulado)


def cortar(pred_cont, umbrales):
    """Convierte la predicción continua en clase 0..4 con los umbrales."""
    return np.digitize(pred_cont, np.sort(umbrales))

In [ ]:
# Parámetros ganadores de v2 (kappa OOF 0,3426 sobre 12 combinaciones). Si cambiás la grilla, pegá acá los nuevos.
BEST_PARAMS = dict(n_estimators=300, learning_rate=0.03, num_leaves=15, colsample_bytree=0.6,
                   random_state=SEED, verbose=-1)

In [ ]:
def seleccion_genetica(X, y, groups, crear_modelo, n_pob=20, n_gen=15, p_cruce=0.5,
                       p_mut=None, elite=2, torneo=3, seed=SEED):
    """Algoritmo genético sobre subconjuntos de columnas de X.

    Cada individuo es una máscara booleana (True = la columna entra al modelo).
    Fitness: kappa out-of-fold con umbrales por cuantiles, igual que en v3.
    Devuelve la lista de columnas ganadora, su kappa y la historia por generación.
    """
    rng = np.random.default_rng(seed)
    cols = np.array(X.columns)
    n = len(cols)
    p_mut = p_mut or 1 / n   # en promedio se da vuelta una columna por hijo

    # mismos folds para todos los individuos, así las comparaciones son justas
    folds = list(StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED).split(X, y, groups))
    cache = {}   # una máscara repetida no se vuelve a entrenar

    def fitness(mask):
        key = mask.tobytes()
        if key not in cache:
            if mask.sum() == 0:
                cache[key] = -1.0
            else:
                Xm = X.loc[:, mask]
                pred = pd.Series(0.0, index=X.index)
                for tr, va in folds:
                    model = crear_modelo()
                    model.fit(Xm.iloc[tr], y.iloc[tr])
                    pred.iloc[va] = model.predict(Xm.iloc[va])
                cache[key] = kappa(y, cortar(pred, umbrales_por_cuantiles(pred, y)))
        return cache[key]

    def elegir(pob, fit):
        # torneo: se sortean `torneo` individuos y gana el de mayor kappa
        idx = rng.choice(len(pob), torneo, replace=False)
        return pob[idx[np.argmax(fit[idx])]]

    pob = rng.random((n_pob, n)) < 0.7
    pob[0] = True   # el individuo con todas las columnas, como referencia
    pob[1] = ~np.char.startswith(cols.astype(str), "emb_svd_")   # individuo sin componentes del embedding; el elitismo lo conserva
    historia = []

    for g in tqdm(range(n_gen)):
        fit = np.array([fitness(ind) for ind in pob])
        orden = np.argsort(fit)[::-1]
        historia.append({"gen": g, "kappa_max": fit.max(), "kappa_media": fit.mean(),
                         "n_cols_mejor": pob[orden[0]].sum()})

        nueva = [pob[i].copy() for i in orden[:elite]]   # elitismo: los mejores pasan sin cambios
        while len(nueva) < n_pob:
            p1, p2 = elegir(pob, fit), elegir(pob, fit)
            hijo = np.where(rng.random(n) < p_cruce, p1, p2)   # cruce uniforme
            hijo = hijo ^ (rng.random(n) < p_mut)               # mutación: da vuelta columnas
            nueva.append(hijo)
        pob = np.array(nueva)

    fit = np.array([fitness(ind) for ind in pob])
    mejor = pob[np.argmax(fit)]
    return list(cols[mejor]), fit.max(), pd.DataFrame(historia)

In [ ]:
# el algoritmo genético parte de la mejor variante: solo embeddings (kappa OOF 0,3880)
# 20 individuos x 20 generaciones: hasta ~400 evaluaciones de ~5 s, unos 30 minutos
X_ga = X_train[cols_emb_only]
cols_ga, kappa_ga, hist_ga = seleccion_genetica(
    X_ga, y_train, groups, lambda: LGBMRegressor(**BEST_PARAMS), n_pob=20, n_gen=20)

print(f"kappa OOF con {len(cols_ga)} de {X_ga.shape[1]} columnas: {kappa_ga:.4f}")
print("columnas que quedaron afuera:", sorted(set(X_ga.columns) - set(cols_ga)))
hist_ga

In [ ]:
OPERACIONES = {
    "mas": lambda a, b: a + b,
    "menos": lambda a, b: a - b,
    "por": lambda a, b: a * b,
    "div": lambda a, b: a / (b.abs() + 1),   # abs + 1 para no dividir por cero (las componentes SVD pueden ser negativas)
}


def aplicar_genes(X, genes):
    """Agrega a X una columna nueva por gen. Cada gen es (columna_a, operación, columna_b)."""
    X = X.copy()
    for a, op, b in genes:
        X[f"{a}__{op}__{b}"] = OPERACIONES[op](X[a], X[b])
    return X


def construccion_genetica(X, y, groups, crear_modelo, cols_combinables=None, n_feat=5, n_pob=10, n_gen=3,
                          p_cruce=0.5, p_mut=0.3, elite=2, torneo=3, seed=SEED):
    """Algoritmo genético que CREA columnas: cada individuo es un conjunto de n_feat columnas nuevas
    que combinan dos columnas de cols_combinables con una operación. Las columnas de X quedan siempre.

    Fitness: kappa out-of-fold de X + las columnas nuevas, con umbrales por cuantiles, igual que en v3.
    Devuelve los genes ganadores, su kappa, el kappa de X sola y la historia por generación.
    """
    rng = np.random.default_rng(seed)
    cols = list(cols_combinables if cols_combinables is not None else X.columns)
    ops = list(OPERACIONES)

    # mismos folds para todos los individuos, así las comparaciones son justas
    folds = list(StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED).split(X, y, groups))
    cache = {}   # un conjunto de genes repetido no se vuelve a entrenar

    def gen_al_azar():
        a, b = rng.choice(len(cols), 2, replace=False)
        return (cols[a], ops[rng.integers(len(ops))], cols[b])

    def fitness(genes):
        key = tuple(sorted(genes))
        if key not in cache:
            Xm = aplicar_genes(X, genes)
            pred = pd.Series(0.0, index=X.index)
            for tr, va in folds:
                model = crear_modelo()
                model.fit(Xm.iloc[tr], y.iloc[tr])
                pred.iloc[va] = model.predict(Xm.iloc[va])
            cache[key] = kappa(y, cortar(pred, umbrales_por_cuantiles(pred, y)))
        return cache[key]

    def elegir(pob, fit):
        # torneo: se sortean `torneo` individuos y gana el de mayor kappa
        idx = rng.choice(len(pob), torneo, replace=False)
        return pob[idx[np.argmax(fit[idx])]]

    def mutar(genes):
        # cada gen, con probabilidad p_mut, cambia una de sus tres partes: columna a, operación o columna b
        genes = list(genes)
        for k in range(len(genes)):
            if rng.random() < p_mut:
                a, op, b = genes[k]
                parte = rng.integers(3)
                if parte == 0:
                    a = cols[rng.integers(len(cols))]
                elif parte == 1:
                    op = ops[rng.integers(len(ops))]
                else:
                    b = cols[rng.integers(len(cols))]
                genes[k] = (a, op, b)
        return genes

    kappa_base = fitness([])   # X sin columnas nuevas, como referencia
    pob = [[gen_al_azar() for _ in range(n_feat)] for _ in range(n_pob)]
    historia = []

    for g in tqdm(range(n_gen)):
        fit = np.array([fitness(ind) for ind in pob])
        orden = np.argsort(fit)[::-1]
        historia.append({"gen": g, "kappa_max": fit.max(), "kappa_media": fit.mean(),
                         "mejora_sobre_base": fit.max() - kappa_base})

        nueva = [pob[i] for i in orden[:elite]]   # elitismo: los mejores pasan sin cambios
        while len(nueva) < n_pob:
            p1, p2 = elegir(pob, fit), elegir(pob, fit)
            hijo = [p1[k] if rng.random() < p_cruce else p2[k] for k in range(n_feat)]   # cruce uniforme
            nueva.append(mutar(hijo))
        pob = nueva

    fit = np.array([fitness(ind) for ind in pob])
    mejor = pob[np.argmax(fit)]
    return mejor, fit.max(), kappa_base, pd.DataFrame(historia)

In [ ]:
import matplotlib.pyplot as plt

# base: solo embeddings (46 columnas, kappa OOF 0,3880). Las 46 quedan siempre;
# las fórmulas solo combinan las 26 columnas interpretables (sin las componentes emb_svd_*)
X_base = X_train[cols_emb_only]
cols_combinables = [c for c in cols_emb_only if not c.startswith("emb_svd_")]

# cuántas columnas nuevas conviene agregar: se corre el algoritmo genético una vez por cada n_feat
VALORES_N = [1, 2, 3, 5, 8, 10]
resultados_n = []
for n in VALORES_N:
    genes_n, kappa_n, kappa_base, _ = construccion_genetica(
        X_base, y_train, groups, lambda: LGBMRegressor(**BEST_PARAMS), cols_combinables=cols_combinables,
        n_feat=n, n_pob=10, n_gen=3)
    resultados_n.append({"n_feat": n, "kappa_oof": kappa_n, "genes": genes_n,
                         "columnas_nuevas": [f"{a} {op} {b}" for a, op, b in genes_n]})
resultados_n = pd.DataFrame(resultados_n)

plt.plot(resultados_n["n_feat"], resultados_n["kappa_oof"], marker="o", label="base + n columnas nuevas")
plt.axhline(kappa_base, color="gray", linestyle="--", label=f"base sola ({kappa_base:.4f})")
plt.xlabel("columnas nuevas agregadas (n_feat)")
plt.ylabel("kappa OOF")
plt.xticks(VALORES_N)
plt.legend()
plt.show()

resultados_n.drop(columns="genes")

## Tres variantes, mismo modelo

In [ ]:
def correr_y_cortar(X, X_t, etiqueta):
    valid_preds, test_preds = correr_cv(lambda: LGBMRegressor(**BEST_PARAMS), X, y_train, X_t, groups)
    umbrales = umbrales_por_cuantiles(valid_preds, y_train)
    k = kappa(y_train, cortar(valid_preds, umbrales))
    print(f"{etiqueta:<26} kappa OOF {k:.4f}  ({X.shape[1]} features)")
    return {"valid_preds": valid_preds, "test_preds": test_preds, "umbrales": umbrales, "kappa": k, "etiqueta": etiqueta}


# las columnas nuevas del mejor n_feat del algoritmo genético, aplicadas igual a train y a test
genes_mejor = resultados_n.loc[resultados_n["kappa_oof"].idxmax(), "genes"]
X_train_gp = aplicar_genes(X_train[cols_emb_only], genes_mejor)
X_test_gp = aplicar_genes(X_test[cols_emb_only], genes_mejor)

variantes = [
    correr_y_cortar(X_train[cols_tfidf], X_test[cols_tfidf], "solo TF-IDF (v4)"),
    correr_y_cortar(X_train[cols_emb_only], X_test[cols_emb_only], "solo embeddings"),
    correr_y_cortar(X_train, X_test, "TF-IDF + embeddings"),
    correr_y_cortar(X_train[cols_ga], X_test[cols_ga], "genético: selección"),
    correr_y_cortar(X_train_gp, X_test_gp, f"genético: +{len(genes_mejor)} columnas"),
]

# completo, para copiar al registro (REGISTRO_GENETICO.md)
print()
print("genético: selección, columnas que quedaron afuera:", sorted(set(cols_emb_only) - set(cols_ga)))
print("genético: columnas nuevas:", [f"{a} {op} {b}" for a, op, b in genes_mejor])
pd.DataFrame([{"variante": v["etiqueta"], "kappa_oof": round(v["kappa"], 4)} for v in variantes])

## Submission con la mejor variante

In [ ]:
mejor = max(variantes, key=lambda v: v["kappa"])
print("se envía:", mejor["etiqueta"])
test_preds, umbrales = mejor["test_preds"], mejor["umbrales"]

# promedio de las 5 predicciones continuas de test, cortado con los MISMOS umbrales calibrados en out-of-fold
submission = pd.Series(cortar(test_preds.mean(axis=1), umbrales), index=X_test.index, name=label)
submission.to_csv("submission.csv")
print(submission.value_counts(normalize=True).sort_index().round(3))
submission.head()

## Tabla de resultados del laboratorio

Completar a mano después de correr y enviar cada versión.

| Versión | Qué agrega | Kappa OOF | LB público | Notas |
|---|---|---|---|---|
| v0 | Baseline de la clase 1: LightGBM sobre numéricas, KFold | | | |
| v1 | Validación por grupo + 3 features de la consigna | | | |
| v2 | Grid search de LightGBM elegido por kappa out-of-fold | | | |
| v3 | Regresión + umbrales por cuantiles | | | |
| v4 | Texto: TF-IDF + SVD y estadísticas de la descripción | | | |
| v5 | Embeddings multilingües de la descripción | | | ← esta notebook |

## Para pensar

- Si "solo embeddings" y "solo TF-IDF" dan parecido pero juntos suben, están capturando cosas distintas. ¿Cuáles?
- Probar `N_EMB` en 10 y 50, y pasar los 384 números crudos sin SVD. LightGBM con 384 columnas más: ¿mejora, empeora, tarda?
- Siguiente escalón de la escalera: las imágenes, con el mismo patrón de dos notebooks (v6a extrae features con una CNN preentrenada, v6b modela).